# Silent reading to imagined speech
This experiment tests whether pretraining on silent-reading EEG improves imagined-speech decoding.
Five sentence-level folds compare imagined-speech training from scratch with reading-to-imagined
fine-tuning at nested 30%, 60%, and 100% training subsets. A shuffled reading condition provides a control.

Set the subject paths and `READING_LABEL_MODES` below, then run the cells in order on a GPU.
Use `["true"]` and `["shuffled"]` in separate sessions, or select both to compare them in one run.
Both checkpoints, all metrics, and the plots are retained. Results and checkpoint weights stay in memory;
restarting the kernel starts a fresh experiment. Tables and figures appear in the output cells.

Dependencies: TensorFlow, NumPy, pandas, scikit-learn, sentence-transformers, openpyxl, matplotlib, and tqdm.
The original run used Python 3.12.13 and TensorFlow 2.19.0 on a Kaggle P100 GPU.

## 1. Setup

In [ ]:
import os
# Set TensorFlow flags before importing it.
GLOBAL_SEED = 1337
os.environ["PYTHONHASHSEED"] = str(GLOBAL_SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"
import gc
import hashlib
import math
import pickle
import random
import re
import time
import textwrap
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from tensorflow.keras import Model, layers
from tqdm.auto import tqdm

def derive_seed(*parts, base_seed=GLOBAL_SEED):
    """Derive a repeatable seed from run names, folds, or other identifiers."""
    payload = "|".join(map(str, (base_seed, *parts))).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:4], "little") & 0x7FFFFFFF

def seed_everything(seed=GLOBAL_SEED):
    random.seed(int(seed))
    np.random.seed(int(seed))
    tf.keras.utils.set_random_seed(int(seed))

def stateless_seed(*parts):
    return np.asarray(
        [derive_seed("stateless_a", *parts), derive_seed("stateless_b", *parts)],
        dtype=np.int32,
    )
seed_everything()
try:
    tf.config.experimental.enable_op_determinism()
except Exception as exc:
    print("Could not explicitly enable TensorFlow determinism:", exc)
try:
    for gpu in tf.config.list_physical_devices("GPU"):
        tf.config.experimental.set_memory_growth(gpu, True)
except Exception as exc:
    print("Could not set GPU memory growth:", exc)
tf.config.optimizer.set_jit(False)
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Global seed:", GLOBAL_SEED)

## 2. Configuration
The defaults match the original experiment. Run names are kept because they determine random seeds.
The true and shuffled conditions therefore use different training seeds, with the same data splits.

In [ ]:
SUBJECT_ID = "sub-02"
SUBJECT_CONFIGS = {
    "sub-01": {
        "reading_root": "/kaggle/input/datasets/shahryarnamdari/chisco-reading-sub01/chisco_sub01_reading_ready",
        "imagined_root": "/kaggle/input/datasets/shahryarnamdari/chisco-is-sub01/chisco_sub01_ready",
    },
    "sub-02": {
        "reading_root": "/kaggle/input/datasets/shahryarnamdari/chisco-reading-sub02/Chisco-Reading-sub02",
        "imagined_root": "/kaggle/input/datasets/shahryarnamdari/chisco-is-sub02/Chisco-IS-sub02",
    },
    "sub-03": {
        "reading_root": "/kaggle/input/datasets/shahryarnamdari/chisco-reading-sub03/Chisco-Reading-sub03",
        "imagined_root": "/kaggle/input/datasets/shahryarnamdari/chisco-is-sub03/Chisco-IS-sub03",
    },
    "sub-04": {
        "reading_root": "/kaggle/input/datasets/shahryarnamdari/chisco-reading-sub04/Chisco-Reading-sub04",
        "imagined_root": "/kaggle/input/datasets/shahryarnamdari/chisco-is-sub04/Chisco-IS-sub04",
    },
    "sub-05": {
        "reading_root": "/kaggle/input/datasets/shahryarnamdari/chisco-reading-sub05/Chisco-Reading-sub05",
        "imagined_root": "/kaggle/input/datasets/shahryarnamdari/chisco-is-sub05/Chisco-IS-sub05",
    },
}
READING_LABEL_MODES = ["true", "shuffled"]  # ["true"], ["shuffled"], or ["true", "shuffled"]
FINETUNE_PROPORTIONS = (0.30, 0.60, 1.00)
INIT_READING_CHECKPOINT = "best_val_top5"
N_FOLDS = 5
VAL_RATIO_OF_TRAIN = 0.15
FOLDS_TO_RUN = list(range(N_FOLDS))
NUM_CLASSES = 39
TARGET_C = 122
TARGET_T = 1651
DROP_LAST_N_CHANNELS = 3
PER_TRIAL_ZSCORE = True
CROP_MODE_BY_PHASE = {"reading": "left", "imagined": "center"}
EEG_DTYPE = np.float16
TEXT_MODEL_NAME = "BAAI/bge-small-zh-v1.5"
TEXT_DEVICE = "cpu"
TRAIN_BATCH_SIZE = 64
EVAL_BATCH_SIZE = 128
MAX_EPOCHS = 100
LEARNING_RATE = 3e-4
EARLY_STOPPING_PATIENCE = 16
MIN_DELTA = 1e-3
REDUCE_LR_PATIENCE = 5
REDUCE_LR_FACTOR = 0.7
MIN_LR = 1e-6
INITIAL_TAU = 0.07
TAU_MIN = 0.01
TAU_MAX = 1.0
LAMBDA_INST = 1.0
LAMBDA_CLS = 0.5
LABEL_SMOOTHING = 0.1
SEMANTIC_NEGATIVE_FLOOR = 0.05
AUG_NOISE_STD = 0.05
AUG_CHANNEL_DROPOUT_RATE = 0.10
AUG_CHANNEL_DROPOUT_PROB = 0.10
AUG_MAX_TEMPORAL_SHIFT = 50
TOPK_LIST = (1, 2, 3, 5, 10)
N_RANDOM_BASELINE = 10_000
N_2V2_PAIRS = 20_000
N_2V2_PERMUTATIONS = 500
if SUBJECT_ID not in SUBJECT_CONFIGS:
    raise KeyError(f"Missing SUBJECT_CONFIGS entry for {SUBJECT_ID}")
if not READING_LABEL_MODES or not set(READING_LABEL_MODES) <= {"true", "shuffled"}:
    raise ValueError("READING_LABEL_MODES must contain 'true', 'shuffled', or both")
if INIT_READING_CHECKPOINT not in {"best_val_loss", "best_val_top5"}:
    raise ValueError("INIT_READING_CHECKPOINT must be best_val_loss or best_val_top5")
PHASES = {
    "reading": {
        "root": Path(SUBJECT_CONFIGS[SUBJECT_ID]["reading_root"]),
        "file_keyword": "task-read",
    },
    "imagined": {
        "root": Path(SUBJECT_CONFIGS[SUBJECT_ID]["imagined_root"]),
        "file_keyword": "task-imagine",
    },
}
print("Subject:", SUBJECT_ID, "| Reading labels:", READING_LABEL_MODES)

## 3. Match EEG trials to sentences and categories
Trials are matched by run, sentence, and occurrence. A sentence-only fallback is used
for unmatched metadata. Category order defines the 39 class IDs.

In [ ]:
chinese_to_english = {
    "预订和旅行安排": "Travel Arrangements",
    "住房和设施": "Housing and Facilities",
    "自然和天气": "Nature and Weather",
    "个人行为和日常活动": "Personal Behavior and Daily Activities",
    "金融和付款": "Finance",
    "饮食和用餐": "Food and Dining",
    "旅行和行李管理": "Travel Affairs",
    "交通和出行": "Transportation and Commuting",
    "旅游和度假": "Vacation",
    "设备故障和环境问题": "Equipment Malfunction or Environmental Issues",
    "价格和费用": "Prices and Costs",
    "时间和日程安排": "Time and Scheduling",
    "衣物和服饰": "Clothing",
    "语言和学习": "Learning",
    "欢迎和感谢": "Welcoming and Thanking",
    "道歉和请求原谅": "Apologies",
    "个人信息": "Personal Information",
    "询问和个人事务": "Inquiries and Personal Matters",
    "饮食习惯": "Eating Habits",
    "产品和质量保证": "Products and Quality Assurance",
    "健康和安全建议": "Health and Safety",
    "家庭关系和家庭事件": "Family Relationships and Events",
    "提供和请求帮助": "Providing and Requesting Assistance",
    "理发和美容护理": "Hairdressing and Beauty Care",
    "节日和庆祝活动": "Festivals and Celebrations",
    "互联网和信息技术": "Internet and Information Technology",
    "健康和身体不适": "Physical Discomfort",
    "情感和人际关系": "Emotions and Interpersonal Relationships",
    "社交和聚会活动": "Gathering Activities",
    "人际交往和情感表达": "Social Interactions",
    "问候和情感状态": "Greetings",
    "工作和职场交流": "Work",
    "表演艺术": "Performing Arts",
    "教育和学习": "Education",
    "娱乐和媒体消费": "Entertainment and Media Consumption",
    "求职和职业发展": "Job Hunting and Career Development",
    "健身": "Fitness",
    "兴趣爱好": "Hobbies",
    "体育和运动": "Sports",
}
CLASS_PHRASES_ZH = list(chinese_to_english)
CLASS2ID = {label: i for i, label in enumerate(CLASS_PHRASES_ZH)}
assert len(CLASS_PHRASES_ZH) == NUM_CLASSES

def numeric_run_id(path):
    name = Path(path).name
    patterns = [r"split_data_(\d+)\.xlsx$", r"run-(\d+)_eeg(?:\s*\(\d+\))?\.pkl$"]
    for pattern in patterns:
        match = re.search(pattern, name)
        if match:
            return int(match.group(1))
    raise ValueError(f"Cannot parse run id from {path}")

def resolve_phase(phase):
    root = PHASES[phase]["root"]
    pkl_dir = root / "derivatives" / "preprocessed_pkl"
    text_dir = root / "textdataset"
    if not pkl_dir.exists() or not text_dir.exists():
        raise FileNotFoundError(f"Missing dataset directories under {root}")
    pkl_files = sorted(
        [p for p in pkl_dir.rglob("*.pkl") if PHASES[phase]["file_keyword"] in p.name],
        key=numeric_run_id,
    )
    if not pkl_files:
        raise RuntimeError(f"No {phase} pickle files found under {pkl_dir}")
    return {"root": root, "text_dir": text_dir, "pkl_files": pkl_files}

def load_text_metadata(phase_info):
    frames = []
    for path in sorted(phase_info["text_dir"].glob("*.xlsx"), key=numeric_run_id):
        frame = pd.read_excel(path)
        sentence_col = "句子" if "句子" in frame else frame.columns[0]
        label_col = "标签" if "标签" in frame else frame.columns[1]
        frame = frame[[sentence_col, label_col]].copy()
        frame.columns = ["sentence_zh", "label_zh"]
        frame["sentence_zh"] = frame["sentence_zh"].astype(str).str.strip()
        frame["label_zh"] = frame["label_zh"].astype(str).str.strip()
        frame["run_id"] = numeric_run_id(path)
        frame["trial_in_run"] = np.arange(len(frame))
        frames.append(frame)
    metadata = pd.concat(frames, ignore_index=True)
    metadata["label_id"] = metadata["label_zh"].map(CLASS2ID)
    if metadata["label_id"].isna().any():
        unknown = metadata.loc[metadata["label_id"].isna(), "label_zh"].unique()
        raise ValueError(f"Unknown labels: {unknown}")
    metadata["label_id"] = metadata["label_id"].astype(int)
    metadata["label_en"] = metadata["label_zh"].map(chinese_to_english)
    return metadata

def add_occurrence(frame):
    frame = frame.copy()
    frame["sentence_occurrence_in_run"] = frame.groupby(
        ["run_id", "sentence_zh"]
    ).cumcount()
    return frame

def load_pickle_list(path):
    with open(path, "rb") as handle:
        samples = pickle.load(handle)
    if not isinstance(samples, list):
        raise TypeError(f"Expected list in {path}, got {type(samples)}")
    return samples

def build_trial_index(phase, phase_info, metadata):
    rows = []
    for path in tqdm(phase_info["pkl_files"], desc=f"index {phase}"):
        samples = load_pickle_list(path)
        run_id = numeric_run_id(path)
        rows.extend({
            "run_id": run_id,
            "eeg_trial_in_pkl": i,
            "sentence_zh": str(sample["text"]).strip(),
            "pkl_path": str(path),
        } for i, sample in enumerate(samples))
    eeg = add_occurrence(pd.DataFrame(rows))
    metadata = add_occurrence(metadata)
    keys = ["run_id", "sentence_zh", "sentence_occurrence_in_run"]
    columns = keys + ["trial_in_run", "label_zh", "label_en", "label_id"]
    index = eeg.merge(metadata[columns], on=keys, how="left", validate="one_to_one")
    missing = index["label_id"].isna()
    if missing.any():
        fallback = metadata.drop_duplicates("sentence_zh").set_index("sentence_zh")
        for column in ["trial_in_run", "label_zh", "label_en", "label_id"]:
            index.loc[missing, column] = index.loc[missing, "sentence_zh"].map(fallback[column])
    if index["label_id"].isna().any():
        raise ValueError(f"EEG sentences missing from text metadata in {phase}")
    index["label_id"] = index["label_id"].astype(int)
    index["trial_uid"] = np.arange(len(index), dtype=np.int64)
    return index
phase_info = {phase: resolve_phase(phase) for phase in PHASES}
text_metadata = {phase: load_text_metadata(phase_info[phase]) for phase in PHASES}
trial_index = {
    phase: build_trial_index(phase, phase_info[phase], text_metadata[phase])
    for phase in PHASES
}
for phase, frame in trial_index.items():
    print(phase, "trials:", len(frame), "sentences:", frame.sentence_zh.nunique(),
          "classes:", frame.label_id.nunique())

## 4. Prepare EEG and text embeddings
Each trial is z-scored across all its channels and samples before cropping. Reading uses a left crop;
imagined speech uses a center crop. EEG is stored in RAM as float16 and converted to float32 in batches.
The text encoder is frozen.

In [ ]:
def to_channels_time(sample_input):
    x = np.asarray(sample_input)
    if x.ndim == 3 and x.shape[0] == 1:
        x = np.squeeze(x, axis=0)
    if x.ndim == 3 and x.shape[-1] == 1:
        x = np.squeeze(x, axis=-1)
    if x.ndim != 2:
        raise ValueError(f"Unexpected EEG shape: {x.shape}")
    if x.shape[0] > 300 and x.shape[1] <= 512:
        x = x.T
    return x.astype(np.float32, copy=False)

def crop_or_pad(x, target_len, mode):
    length = x.shape[1]
    if length > target_len:
        starts = {"left": 0, "center": (length - target_len) // 2,
                  "right": length - target_len}
        if mode not in starts:
            raise ValueError(f"Unknown crop mode: {mode}")
        start = starts[mode]
        return x[:, start:start + target_len]
    if length < target_len:
        total = target_len - length
        left = total // 2
        return np.pad(x, ((0, 0), (left, total - left)), mode="constant")
    return x

def preprocess_eeg(sample_input, phase):
    """Match channels, normalize the full trial, then crop or pad in time."""
    x = to_channels_time(sample_input)
    if x.shape[0] >= TARGET_C + DROP_LAST_N_CHANNELS:
        x = x[:-DROP_LAST_N_CHANNELS]
    else:
        x = x[:min(TARGET_C, x.shape[0])]
        if x.shape[0] < TARGET_C:
            x = np.pad(x, ((0, TARGET_C - x.shape[0]), (0, 0)))
    if x.shape[0] != TARGET_C:
        raise ValueError(f"Expected {TARGET_C} channels, got {x.shape}")
    if PER_TRIAL_ZSCORE:
        x = (x - x.mean(keepdims=True)) / (x.std(keepdims=True) + 1e-6)
    x = crop_or_pad(x, TARGET_T, CROP_MODE_BY_PHASE[phase])
    return np.nan_to_num(x).astype(np.float32)

def l2_normalize_np(x, axis=1, eps=1e-12):
    x = np.asarray(x, dtype=np.float32)
    return x / np.maximum(np.linalg.norm(x, axis=axis, keepdims=True), eps)

def load_eeg(phase):
    """Preprocess each trial once and store it in RAM as float16."""
    frame = trial_index[phase]
    eeg = np.empty((len(frame), TARGET_C, TARGET_T), dtype=EEG_DTYPE)
    for path, group in tqdm(frame.groupby("pkl_path", sort=False), desc=f"EEG {phase}"):
        samples = load_pickle_list(path)
        for row in group.itertuples():
            eeg[row.trial_uid] = preprocess_eeg(
                samples[row.eeg_trial_in_pkl]["input_features"], phase
            ).astype(EEG_DTYPE)
        del samples
    gc.collect()
    return eeg
eeg_data = {phase: load_eeg(phase) for phase in PHASES}
all_sentences = sorted(set().union(*[
    set(frame.sentence_zh) for frame in trial_index.values()
]))

def encode_text():
    """Embed unique sentences and category names with the frozen text model."""
    from sentence_transformers import SentenceTransformer
    text_model = SentenceTransformer(TEXT_MODEL_NAME, device=TEXT_DEVICE)
    sentence_emb = text_model.encode(
        all_sentences, batch_size=256, convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=True,
    ).astype(np.float32)
    class_emb = text_model.encode(
        CLASS_PHRASES_ZH, batch_size=256, convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=True,
    ).astype(np.float32)
    return l2_normalize_np(sentence_emb), l2_normalize_np(class_emb)
sentence_emb, class_emb = encode_text()
sentence_to_emb = dict(zip(all_sentences, sentence_emb))
targets = {}
for phase, frame in trial_index.items():
    targets[phase] = (
        np.vstack([sentence_to_emb[s] for s in frame.sentence_zh]).astype(np.float32),
        frame.label_id.to_numpy(np.int64),
    )
EMB_DIM = int(class_emb.shape[1])
CLASS_PROTOTYPES = tf.constant(class_emb, dtype=tf.float32)
print("Embedding dimension:", EMB_DIM)

## 5. Shared-sentence cross-validation
Only sentences present in both phases with matching category labels are included. Train, validation,
and test sentences are disjoint. Scratch and transfer runs use the same nested training subsets.

In [ ]:
def common_sentence_table():
    tables = []
    for phase, frame in trial_index.items():
        table = frame[["sentence_zh", "label_id"]].drop_duplicates()
        if table.groupby("sentence_zh").label_id.nunique().max() != 1:
            raise ValueError(f"A {phase} sentence maps to multiple classes")
        tables.append(table.rename(columns={"label_id": f"label_{phase}"}))
    common = tables[0].merge(tables[1], on="sentence_zh", how="inner")
    common = common[common.label_reading == common.label_imagined].copy()
    common["label_id"] = common.label_reading.astype(int)
    return common[["sentence_zh", "label_id"]].drop_duplicates().sort_values(
        "sentence_zh"
    ).reset_index(drop=True)
COMMON_TABLE = common_sentence_table()

def nested_subsets(train_frame, fold_id):
    """Shuffle sentences within each category and take nested prefixes."""
    subsets = {proportion: [] for proportion in FINETUNE_PROPORTIONS}
    for class_id in range(NUM_CLASSES):
        sentences = sorted(train_frame.loc[
            train_frame.label_id == class_id, "sentence_zh"
        ].unique())
        rng = np.random.default_rng(derive_seed("nested", fold_id, class_id))
        ordered = np.asarray(sentences, dtype=object)[rng.permutation(len(sentences))]
        for proportion in FINETUNE_PROPORTIONS:
            n = max(1, min(len(ordered), int(round(len(ordered) * proportion))))
            subsets[proportion].extend(ordered[:n].tolist())
    return {proportion: set(sentences) for proportion, sentences in subsets.items()}

def build_splits():
    sentences = COMMON_TABLE.sentence_zh.to_numpy()
    labels = COMMON_TABLE.label_id.to_numpy()
    outer = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=GLOBAL_SEED)
    splits = []
    for fold_id, (trainval_pos, test_pos) in enumerate(outer.split(sentences, labels)):
        trainval = COMMON_TABLE.iloc[trainval_pos]
        inner = StratifiedShuffleSplit(
            n_splits=1, test_size=VAL_RATIO_OF_TRAIN, random_state=GLOBAL_SEED
        )
        train_rel, val_rel = next(inner.split(trainval.sentence_zh, trainval.label_id))
        train_frame = trainval.iloc[train_rel]
        split = {
            "fold_id": fold_id,
            "train_sents": set(train_frame.sentence_zh),
            "val_sents": set(trainval.iloc[val_rel].sentence_zh),
            "test_sents": set(COMMON_TABLE.iloc[test_pos].sentence_zh),
            "subsets": nested_subsets(train_frame, fold_id),
        }
        splits.append(split)
    return splits

def validate_splits(splits):
    common = set(COMMON_TABLE.sentence_zh)
    pooled_test = []
    for split in splits:
        train, val, test = split["train_sents"], split["val_sents"], split["test_sents"]
        assert not (train & val or train & test or val & test)
        assert train | val | test == common
        previous = set()
        for proportion in sorted(FINETUNE_PROPORTIONS):
            current = split["subsets"][proportion]
            assert previous <= current <= train
            previous = current
        pooled_test.extend(test)
    if len(pooled_test) != len(common) or set(pooled_test) != common:
        raise AssertionError("Each shared sentence must occur in exactly one outer test fold")

def indices_for_sentences(phase, sentences):
    return np.flatnonzero(trial_index[phase].sentence_zh.isin(sentences).to_numpy()).astype(np.int64)
CV_SPLITS = build_splits()
validate_splits(CV_SPLITS)
split_summary = []
for split in CV_SPLITS:
    for name in ["train", "val", "test"]:
        split_summary.append({
            "fold_id": split["fold_id"], "split": name,
            "sentences": len(split[f"{name}_sents"]),
            "reading_trials": len(indices_for_sentences("reading", split[f"{name}_sents"])),
            "imagined_trials": len(indices_for_sentences("imagined", split[f"{name}_sents"])),
        })
display(pd.DataFrame(split_summary))

## 6. EEG CNN–Transformer encoder

In [ ]:
class SqueezeChannels(layers.Layer):
    def call(self, inputs):
        return tf.squeeze(inputs, axis=1)

class L2Normalize(layers.Layer):
    def call(self, inputs):
        return tf.math.l2_normalize(inputs, axis=-1)

def chisco_eeg_feature_extractor(inputs, dropout_rate=0.5):
    x = layers.Conv2D(8, (1, 125), padding="same", use_bias=False)(inputs)
    x = layers.LayerNormalization(axis=-1, epsilon=1e-6)(x)
    x = layers.DepthwiseConv2D((TARGET_C, 1), use_bias=False, depth_multiplier=8)(x)
    x = layers.LayerNormalization(axis=-1, epsilon=1e-6)(x)
    x = layers.Activation("elu")(x)
    x = layers.AveragePooling2D((1, 2))(x)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.SeparableConv2D(64, (1, 25), padding="same", use_bias=False)(x)
    x = layers.LayerNormalization(axis=-1, epsilon=1e-6)(x)
    x = layers.Activation("elu")(x)
    x = layers.AveragePooling2D((1, 5))(x)
    x = layers.Dropout(dropout_rate)(x)
    return SqueezeChannels()(x)

class PositionalEncoding(layers.Layer):
    def call(self, inputs):
        length, width = tf.shape(inputs)[1], tf.shape(inputs)[2]
        position = tf.cast(tf.range(length)[:, None], tf.float32)
        divisor = tf.exp(
            tf.range(0, width, 2, dtype=tf.float32)
            * -(math.log(10000.0) / tf.cast(width, tf.float32))
        )
        encoding = tf.concat(
            [tf.sin(position * divisor), tf.cos(position * divisor)], axis=-1
        )[:, :width]
        return inputs + encoding[None, :, :]

def transformer_block(inputs, dropout=0.1):
    attention = layers.MultiHeadAttention(
        num_heads=8, key_dim=64, dropout=dropout
    )(inputs, inputs)
    x = layers.LayerNormalization(epsilon=1e-6)(inputs + attention)
    feedforward = layers.Dense(256, activation="gelu")(x)
    feedforward = layers.Dropout(dropout)(feedforward)
    feedforward = layers.Dense(64)(feedforward)
    return layers.LayerNormalization(epsilon=1e-6)(x + feedforward)

class AttentionPooling1D(layers.Layer):
    def build(self, input_shape):
        self.weight = self.add_weight(
            name="att_weight", shape=(input_shape[-1], 1),
            initializer="glorot_uniform", trainable=True,
        )
        self.bias = self.add_weight(
            name="att_bias", shape=(1,), initializer="zeros", trainable=True
        )
        super().build(input_shape)
    def call(self, inputs):
        scores = tf.squeeze(tf.tanh(tf.matmul(inputs, self.weight) + self.bias), -1)
        weights = tf.nn.softmax(scores, axis=1)
        return tf.reduce_sum(inputs * weights[..., None], axis=1)

def build_eeg_encoder(cnn_dropout=0.5, transformer_dropout=0.1):
    inputs = layers.Input((TARGET_C, TARGET_T, 1), name="eeg_input")
    x = chisco_eeg_feature_extractor(inputs, cnn_dropout)
    x = layers.Dropout(transformer_dropout)(PositionalEncoding()(x))
    for _ in range(4):
        x = transformer_block(x, transformer_dropout)
    x = AttentionPooling1D()(x)
    x = layers.Dense(256, activation="gelu")(x)
    x = layers.Dropout(transformer_dropout)(x)
    x = layers.Dense(EMB_DIM, name="projection_head")(x)
    outputs = L2Normalize(name="eeg_emb_norm")(x)
    return Model(inputs, outputs, name="Chisco_Contrastive_Encoder")
tf.keras.backend.clear_session()
seed_everything(derive_seed("architecture"))
encoder_preview = build_eeg_encoder()
encoder_preview.summary()
del encoder_preview
gc.collect()

## 7. Losses, augmentation, and batches
Training combines instance contrastive loss and category classification loss. The temperature remains
fixed at its initial value. Validation batches use the training
batch size because the instance loss depends on the other examples in the batch.

The shuffled control permutes reading training sentences once per run, moving sentence embeddings and
category labels together. Repeated sentences share a replacement. Validation and imagined fine-tuning
keep their correct targets.

In [ ]:
def calculate_class_weights(labels):
    counts = np.bincount(labels, minlength=NUM_CLASSES).astype(np.float32)
    frequency = (counts + 1e-6) / (counts.sum() + 1e-6)
    weights = (1.0 / np.sqrt(frequency))
    return tf.constant((weights / weights.mean()).astype(np.float32)), counts.astype(int)

@tf.function
def l2n_tf(x):
    return tf.math.l2_normalize(tf.cast(x, tf.float32), axis=-1)

@tf.function
def sample_weights(labels, class_weights):
    weights = tf.gather(class_weights, labels)
    return weights / (tf.reduce_mean(weights) + 1e-8)

@tf.function
def augment_eeg(x, seed):
    x = tf.cast(x, tf.float32)
    seeds = tf.random.experimental.stateless_split(tf.cast(seed, tf.int32), 4)
    x += tf.random.stateless_normal(tf.shape(x), seeds[0], stddev=AUG_NOISE_STD)
    apply_dropout = tf.random.stateless_uniform([], seeds[1]) < AUG_CHANNEL_DROPOUT_PROB
    def drop_channels():
        shape = tf.stack([tf.shape(x)[0], TARGET_C, 1, 1])
        keep = tf.cast(
            tf.random.stateless_uniform(shape, seeds[2]) > AUG_CHANNEL_DROPOUT_RATE,
            tf.float32,
        )
        return x * keep
    x = tf.cond(apply_dropout, drop_channels, lambda: x)
    shift = tf.random.stateless_uniform(
        [], seeds[3], minval=-AUG_MAX_TEMPORAL_SHIFT,
        maxval=AUG_MAX_TEMPORAL_SHIFT + 1, dtype=tf.int32,
    )
    return tf.roll(x, shift=shift, axis=2)

@tf.function
def cosine_logits(eeg_z, tau):
    return tf.matmul(l2n_tf(eeg_z), l2n_tf(CLASS_PROTOTYPES), transpose_b=True) / tau

@tf.function
def instance_loss(eeg_z, text_z, labels, class_weights, tau):
    """Align paired embeddings while downweighting semantically similar negatives."""
    eeg_z, text_z = l2n_tf(eeg_z), l2n_tf(text_z)
    logits = tf.matmul(eeg_z, text_z, transpose_b=True) / tau
    semantic_weights = tf.maximum(
        SEMANTIC_NEGATIVE_FLOOR, 1.0 - tf.matmul(text_z, text_z, transpose_b=True)
    )
    eye = tf.eye(tf.shape(eeg_z)[0])
    semantic_weights = eye + (1.0 - eye) * semantic_weights
    maximum = tf.stop_gradient(tf.reduce_max(logits, axis=1, keepdims=True))
    denominator = tf.math.log(
        tf.reduce_sum(semantic_weights * tf.exp(logits - maximum), axis=1) + 1e-9
    ) + tf.squeeze(maximum, axis=1)
    loss = -(tf.linalg.diag_part(logits) - denominator)
    return tf.reduce_mean(loss * sample_weights(labels, class_weights))

@tf.function
def classification_loss(logits, labels, class_weights):
    one_hot = tf.one_hot(labels, NUM_CLASSES)
    loss = tf.keras.losses.categorical_crossentropy(
        one_hot, logits, from_logits=True, label_smoothing=LABEL_SMOOTHING
    )
    return tf.reduce_mean(loss * sample_weights(labels, class_weights))

def deterministic_order(indices, training, seed=None, drop_remainder=False):
    indices = np.asarray(indices, dtype=np.int64)
    order = np.random.default_rng(int(seed)).permutation(indices) if training else indices.copy()
    if drop_remainder:
        order = order[:len(order) // TRAIN_BATCH_SIZE * TRAIN_BATCH_SIZE]
    return order

def iter_batches(phase, indices, text_array, class_array, batch_size,
                 training=False, seed=None, drop_remainder=False):
    order = deterministic_order(indices, training, seed, drop_remainder)
    source = eeg_data[phase]
    for start in range(0, len(order), batch_size):
        batch_indices = order[start:start + batch_size]
        yield (
            source[batch_indices].astype(np.float32, copy=False)[..., None],
            text_array[batch_indices].astype(np.float32, copy=False),
            class_array[batch_indices].astype(np.int64, copy=False),
        )

def training_targets(phase, train_indices, label_mode, run_seed):
    """Shuffle training sentence targets together with their category labels."""
    true_text, true_labels = targets[phase]
    if label_mode == "true":
        return true_text, true_labels
    if phase != "reading" or label_mode != "shuffled":
        raise ValueError("Only reading training supports label_mode='shuffled'")
    train_sentence_values = trial_index[phase].iloc[train_indices].sentence_zh.to_numpy()
    sentences = np.asarray(sorted(set(train_sentence_values)), dtype=object)
    shuffled = sentences[np.random.default_rng(run_seed).permutation(len(sentences))]
    mapping = dict(zip(sentences, shuffled))
    text_out, labels_out = true_text.copy(), true_labels.copy()
    label_by_sentence = trial_index[phase].drop_duplicates("sentence_zh").set_index(
        "sentence_zh"
    ).label_id
    for source_sentence, target_sentence in mapping.items():
        selected = train_indices[train_sentence_values == source_sentence]
        text_out[selected] = sentence_to_emb[target_sentence]
        labels_out[selected] = int(label_by_sentence.loc[target_sentence])
    return text_out, labels_out

## 8. Train one condition
Both best-validation-loss and best-validation-top-5 weights are kept in memory. Early stopping and
learning-rate reductions follow validation loss. Fine-tuning starts with the selected reading weights
and a fresh optimizer. The same function handles scratch, reading, and transfer runs.

In [ ]:
def make_metrics():
    """Create independent accumulators for training or validation."""
    return {
        "total": tf.keras.metrics.Mean(), "instance": tf.keras.metrics.Mean(),
        "class": tf.keras.metrics.Mean(), "top1": tf.keras.metrics.SparseCategoricalAccuracy(),
        "top5": tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5),
    }

def train_run(spec):
    """Train one condition and retain both selected checkpoints in memory."""
    fold_id, run_name, phase = spec["fold_id"], spec["run_name"], spec["phase"]
    train_indices = np.asarray(spec["train_indices"], dtype=np.int64)
    val_indices = np.asarray(spec["val_indices"], dtype=np.int64)
    run_seed = derive_seed("run", SUBJECT_ID, fold_id, run_name)
    train_text, train_labels = training_targets(
        phase, train_indices, spec["label_mode"], derive_seed("shuffle", run_seed)
    )
    val_text, val_labels = targets[phase]
    class_weights, class_counts = calculate_class_weights(train_labels[train_indices])
    tf.keras.backend.clear_session()
    gc.collect()
    seed_everything(run_seed)
    encoder = build_eeg_encoder()
    if spec.get("initial_weights") is not None:
        encoder.set_weights(spec["initial_weights"])
    optimizer = tf.keras.optimizers.Adam(LEARNING_RATE)
    log_temperature = tf.Variable(
        tf.math.log(tf.constant(INITIAL_TAU, tf.float32)), trainable=True,
        name="log_temperature",
    )
    names = ["total", "instance", "class", "top1", "top5"]
    train_metrics, val_metrics = make_metrics(), make_metrics()
    @tf.function(reduce_retracing=True)
    def step(x, text_z, labels, training, augmentation_seed):
        # Temperature stays fixed because this calculation is outside the tape.
        tau = tf.clip_by_value(tf.exp(log_temperature), TAU_MIN, TAU_MAX)
        if training:
            x = augment_eeg(x, augmentation_seed)
        with tf.GradientTape() as tape:
            eeg_z = encoder(x, training=training)
            logits = cosine_logits(eeg_z, tau)
            inst = instance_loss(eeg_z, text_z, labels, class_weights, tau)
            cls = classification_loss(logits, labels, class_weights)
            total = LAMBDA_INST * inst + LAMBDA_CLS * cls
        if training:
            variables = encoder.trainable_variables + [log_temperature]
            gradients = tape.gradient(total, variables)
            optimizer.apply_gradients(
                [(gradient, variable) for gradient, variable in zip(gradients, variables)
                 if gradient is not None]
            )
        metrics = train_metrics if training else val_metrics
        metrics["total"].update_state(total)
        metrics["instance"].update_state(inst)
        metrics["class"].update_state(cls)
        metrics["top1"].update_state(labels, logits)
        metrics["top5"].update_state(labels, logits)
    best_loss, best_top5 = np.inf, -np.inf
    best_lr_loss, bad_early, bad_lr = np.inf, 0, 0
    checkpoints, checkpoint_metadata, history = {}, {}, []
    print(run_name, "| fold", fold_id, "|", phase, "| train/val", len(train_indices), len(val_indices))
    for epoch in range(1, MAX_EPOCHS + 1):
        started = time.time()
        for metric in list(train_metrics.values()) + list(val_metrics.values()):
            metric.reset_state()
        order_seed = derive_seed("batch_order", run_seed, epoch)
        train_iter = iter_batches(
            phase, train_indices, train_text, train_labels, TRAIN_BATCH_SIZE,
            training=True, seed=order_seed, drop_remainder=True,
        )
        for batch_id, (x, text_z, labels) in enumerate(train_iter):
            step(x, text_z, labels, True, stateless_seed("augment", run_seed, epoch, batch_id))
        val_iter = iter_batches(
            phase, val_indices, val_text, val_labels, TRAIN_BATCH_SIZE,
            training=False, drop_remainder=False,
        )
        for batch_id, (x, text_z, labels) in enumerate(val_iter):
            step(x, text_z, labels, False, stateless_seed("validation", run_seed, epoch, batch_id))
        tau = float(tf.clip_by_value(tf.exp(log_temperature), TAU_MIN, TAU_MAX).numpy())
        row = {
            "epoch": epoch, "epoch_order_seed": order_seed,
            **{f"train_{name}": float(train_metrics[name].result().numpy()) for name in names},
            **{f"val_{name}": float(val_metrics[name].result().numpy()) for name in names},
            "tau": tau, "learning_rate": float(optimizer.learning_rate.numpy()),
            "seconds": time.time() - started,
        }
        history.append(row)
        notes = []
        if row["val_total"] < best_loss - MIN_DELTA:
            best_loss, bad_early = row["val_total"], 0
            checkpoints["best_val_loss"] = encoder.get_weights()
            checkpoint_metadata["best_val_loss"] = {
                "epoch": epoch, "metric": best_loss, "tau": tau,
            }
            notes.append("best loss")
        else:
            bad_early += 1
        if row["val_top5"] > best_top5 + MIN_DELTA:
            best_top5 = row["val_top5"]
            checkpoints["best_val_top5"] = encoder.get_weights()
            checkpoint_metadata["best_val_top5"] = {
                "epoch": epoch, "metric": best_top5, "tau": tau,
            }
            notes.append("best top5")
        if row["val_total"] < best_lr_loss - MIN_DELTA:
            best_lr_loss, bad_lr = row["val_total"], 0
        else:
            bad_lr += 1
        if bad_lr >= REDUCE_LR_PATIENCE:
            old_lr = float(optimizer.learning_rate.numpy())
            new_lr = max(old_lr * REDUCE_LR_FACTOR, MIN_LR)
            optimizer.learning_rate.assign(new_lr)
            bad_lr = 0
            notes.append(f"lr {old_lr:.1e}->{new_lr:.1e}")
        print(
            f"Ep {epoch:03d} | loss {row['train_total']:.4f}/{row['val_total']:.4f} | "
            f"top1 {row['train_top1']:.3f}/{row['val_top1']:.3f} | "
            f"top5 {row['train_top5']:.3f}/{row['val_top5']:.3f} | "
            f"tau {tau:.4f} | lr {row['learning_rate']:.1e}"
            + (" | " + "; ".join(notes) if notes else "")
        )
        if bad_early >= EARLY_STOPPING_PATIENCE:
            print("Early stopping at epoch", epoch)
            break
    record = {
        **{key: value for key, value in spec.items()
           if key not in {"train_indices", "val_indices", "initial_weights"}},
        "run_seed": run_seed, "n_train_trials": len(train_indices),
        "n_val_trials": len(val_indices), "class_counts": class_counts.tolist(),
        "checkpoints": checkpoints, "checkpoint_metadata": checkpoint_metadata,
        "history": pd.DataFrame(history),
    }
    del encoder
    gc.collect()
    return record

## 9. Run the experiment
Each fold trains the scratch conditions once, followed by each selected reading condition and its
three transfer runs. Changing `READING_LABEL_MODES` changes which conditions run; it does not change
their names, seeds, splits, or training settings. Training history remains available in `run_records`.

In [ ]:
def make_spec(fold_id, run_name, role, phase, train_sentences, val_sentences,
              label_mode="true", proportion=None, initial_weights=None):
    return {
        "fold_id": int(fold_id), "run_name": run_name, "role": role,
        "phase": phase, "label_mode": label_mode, "proportion": proportion,
        "train_indices": indices_for_sentences(phase, train_sentences),
        "val_indices": indices_for_sentences(phase, val_sentences),
        "initial_weights": initial_weights,
    }

def run_experiment():
    """Run scratch baselines, reading pretraining, and imagined-speech fine-tuning."""
    records = []
    for fold_id in FOLDS_TO_RUN:
        split = CV_SPLITS[fold_id]
        for proportion in FINETUNE_PROPORTIONS:
            spec = make_spec(
                fold_id, f"scratch_imagined_{int(proportion * 100):03d}pct",
                "scratch_imagined", "imagined", split["subsets"][proportion],
                split["val_sents"], proportion=proportion,
            )
            records.append(train_run(spec))
        for label_mode in READING_LABEL_MODES:
            source_spec = make_spec(
                fold_id, f"reading_{label_mode}_shared100", "reading_source",
                "reading", split["train_sents"], split["val_sents"],
                label_mode=label_mode, proportion=1.0,
            )
            source = train_run(source_spec)
            records.append(source)
            for proportion in FINETUNE_PROPORTIONS:
                spec = make_spec(
                    fold_id, f"transfer_{label_mode}_imagined_{int(proportion * 100):03d}pct",
                    "transfer_imagined", "imagined", split["subsets"][proportion],
                    split["val_sents"], label_mode="true", proportion=proportion,
                    initial_weights=source["checkpoints"][INIT_READING_CHECKPOINT],
                )
                spec["source_label_mode"] = label_mode
                records.append(train_run(spec))
    return records
run_records = run_experiment()
display(pd.DataFrame([
    {"fold_id": r["fold_id"], "run_name": r["run_name"], "train_trials": r["n_train_trials"],
     "epochs": len(r["history"]),
     **{f"{name}_epoch": info["epoch"] for name, info in r["checkpoint_metadata"].items()}}
    for r in run_records
]))

## 10. Predict with both checkpoints
Reading models are evaluated on reading EEG and, without fine-tuning, on imagined EEG.
All other conditions are evaluated on imagined EEG. Predictions stay in memory for both metric sections.

In [ ]:
def build_evaluation_jobs(records):
    jobs = []
    for record in records:
        fold_id = int(record["fold_id"])
        split = CV_SPLITS[fold_id]
        for checkpoint in ["best_val_loss", "best_val_top5"]:
            base = {
                "fold_id": fold_id, "run_name": record["run_name"],
                "role": record["role"], "proportion": record.get("proportion"),
                "source_label_mode": record.get("source_label_mode", record.get("label_mode")),
                "checkpoint": checkpoint, "weights": record["checkpoints"][checkpoint],
                "class_counts": record["class_counts"],
            }
            if record["role"] == "reading_source":
                mode = record["label_mode"]
                jobs.extend([
                    {**base, "condition": f"Reading {mode} → Reading test",
                     "eval_phase": "reading",
                     "eval_indices": indices_for_sentences("reading", split["test_sents"])},
                    {**base, "condition": f"Reading {mode} → Imagined zero-shot",
                     "eval_phase": "imagined",
                     "eval_indices": indices_for_sentences("imagined", split["test_sents"])},
                ])
            else:
                if record["role"] == "scratch_imagined":
                    condition = f"Imagined scratch {int(record['proportion'] * 100)}%"
                else:
                    condition = (
                        f"Reading {record['source_label_mode']} transfer → Imagined "
                        f"{int(record['proportion'] * 100)}%"
                    )
                jobs.append({
                    **base, "condition": condition, "eval_phase": "imagined",
                    "eval_indices": indices_for_sentences("imagined", split["test_sents"]),
                })
    return jobs

def predict_embeddings(encoder, phase, indices):
    text_array, label_array = targets[phase]
    output = []
    iterator = iter_batches(
        phase, indices, text_array, label_array, EVAL_BATCH_SIZE,
        training=False, drop_remainder=False,
    )
    for x, _, _ in tqdm(iterator, total=math.ceil(len(indices) / EVAL_BATCH_SIZE),
                        desc=f"predict {phase}", leave=False):
        output.append(encoder(x, training=False).numpy().astype(np.float32))
    return l2_normalize_np(np.vstack(output))

def predict_job(job):
    """Restore a checkpoint and predict embeddings for its held-out trials."""
    tf.keras.backend.clear_session()
    seed_everything(derive_seed("evaluation", job["fold_id"], job["run_name"], job["checkpoint"]))
    encoder = build_eeg_encoder()
    encoder.set_weights(job["weights"])
    indices = np.asarray(job["eval_indices"], dtype=np.int64)
    z = predict_embeddings(encoder, job["eval_phase"], indices)
    text_array, label_array = targets[job["eval_phase"]]
    text_true = l2_normalize_np(text_array[indices])
    y_true = label_array[indices]
    del encoder
    gc.collect()
    return {**{key: value for key, value in job.items() if key != "weights"},
            "z": z, "text_true": text_true, "y_true": y_true}
prediction_items = [predict_job(job) for job in build_evaluation_jobs(run_records)]
print("Evaluation jobs:", len(prediction_items))

## 11. Top-k accuracy and Macro F1
Top-k results use category-embedding similarities. The random baseline draws rankings from the training
class prior over 10,000 runs. Summary error bars show the sample standard deviation across folds.

In [ ]:
def topk_accuracy(y_true, ranked, k):
    return float(np.mean((ranked[:, :k] == y_true[:, None]).any(axis=1)))

def prior_random_topk(y_true, class_counts, ks=TOPK_LIST, n_runs=N_RANDOM_BASELINE,
                      seed=GLOBAL_SEED):
    """Sample class-prior rankings with Gumbel noise and summarize their accuracy."""
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true, dtype=np.int64)
    prior = np.asarray(class_counts, dtype=np.float32)
    prior = prior / prior.sum()
    log_prior = np.log(prior + 1e-12).astype(np.float32)
    ks, max_k = tuple(sorted(ks)), max(ks)
    bytes_per_run = len(y_true) * NUM_CLASSES * np.dtype(np.float32).itemsize
    chunk_size = int(np.clip((80 * 1024**2) // max(bytes_per_run, 1), 20, 500))
    sums, sums_squared, completed = {k: 0.0 for k in ks}, {k: 0.0 for k in ks}, 0
    for start in range(0, n_runs, chunk_size):
        count = min(chunk_size, n_runs - start)
        scores = rng.gumbel(size=(count, len(y_true), NUM_CLASSES)).astype(np.float32)
        scores += log_prior[None, None, :]
        candidates = np.argpartition(scores, -max_k, axis=2)[:, :, -max_k:]
        candidate_scores = np.take_along_axis(scores, candidates, axis=2)
        ranked = np.take_along_axis(candidates, np.argsort(-candidate_scores, axis=2), axis=2)
        for k in ks:
            accuracy = (ranked[:, :, :k] == y_true[None, :, None]).any(axis=2).mean(axis=1)
            sums[k] += float(accuracy.sum())
            sums_squared[k] += float(np.square(accuracy).sum())
        completed += count
    output = {}
    for k in ks:
        mean = sums[k] / completed
        variance = max(sums_squared[k] / completed - mean**2, 0.0)
        output[k] = (float(mean), float(np.sqrt(variance)))
    return output

def fold_std(series):
    """Compute the sample standard deviation over available fold scores."""
    values = pd.Series(series).dropna().to_numpy(float)
    if not len(values):
        return np.nan
    return float(values.std(ddof=1)) if len(values) > 1 else 0.0

def evaluate_topk(items):
    topk_rows, f1_rows = [], []
    prototypes = l2_normalize_np(class_emb)
    random_cache = {}
    for item in tqdm(items, desc="Top-k and Macro F1"):
        logits = l2_normalize_np(item["z"]) @ prototypes.T
        ranked = np.argsort(-logits, axis=1)
        prediction = ranked[:, 0]
        random_key = (
            item["fold_id"], item["eval_phase"], tuple(item["class_counts"]),
            hashlib.sha256(np.asarray(item["y_true"], np.int64).tobytes()).hexdigest(),
        )
        if random_key not in random_cache:
            random_cache[random_key] = prior_random_topk(
                item["y_true"], item["class_counts"],
                seed=derive_seed("topk_random", *random_key[:3]),
            )
        random_stats = random_cache[random_key]
        common = {
            "fold_id": item["fold_id"], "condition": item["condition"],
            "run_name": item["run_name"], "checkpoint": item["checkpoint"],
            "eval_phase": item["eval_phase"], "n_trials": len(item["y_true"]),
        }
        f1_rows.append({
            **common,
            "macro_f1": f1_score(
                item["y_true"], prediction, labels=np.arange(NUM_CLASSES),
                average="macro", zero_division=0,
            ),
        })
        for k in TOPK_LIST:
            topk_rows.append({
                **common, "k": k,
                "topk_accuracy": topk_accuracy(item["y_true"], ranked, k),
                "random_mean": random_stats[k][0], "random_std": random_stats[k][1],
            })
    return pd.DataFrame(topk_rows), pd.DataFrame(f1_rows)

def summarize_folds(frame, groups, aggregations):
    return frame.groupby(groups, as_index=False).agg(n_folds=("fold_id", "nunique"), **aggregations)

def show_table(title, frame):
    print(title)
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(frame)
topk_results, macro_f1_results = evaluate_topk(prediction_items)
topk_summary = summarize_folds(topk_results, ["condition", "checkpoint", "eval_phase", "k"], {
    "topk_mean": ("topk_accuracy", "mean"), "topk_std": ("topk_accuracy", fold_std),
    "random_mean": ("random_mean", "mean"),
})
macro_f1_summary = summarize_folds(macro_f1_results, ["condition", "checkpoint", "eval_phase"], {
    "macro_f1_mean": ("macro_f1", "mean"), "macro_f1_std": ("macro_f1", fold_std),
})
show_table("Top-k accuracy by fold", topk_results)
show_table("Top-k cross-validation summary", topk_summary)
show_table("Macro F1 by fold", macro_f1_results)
show_table("Macro F1 cross-validation summary", macro_f1_summary)

## 12. Sentence-level 2v2 evaluation
Each test compares matched EEG–sentence pairs with swapped pairs; ties score 0.5. The three sampling
modes balance different categories, balance the same category, or sample uniformly over trials.
Seeds depend on fold and mode, giving conditions with the same ordered test trials the same sampled pairs.

Evaluation uses 20,000 observed pairs and 500 null permutations with 5,000 pairs each.
Null targets are permuted globally, including in the same-category mode. The summary's `mean_p_value`
is the arithmetic mean of fold p-values, not a combined significance test.

In [ ]:
TWOVTWO_MODES = (
    "different_category_balanced",
    "same_category_balanced",
    "random_overall_trial_uniform",
)

def indices_by_class(labels):
    return {int(label): np.flatnonzero(labels == label) for label in np.unique(labels)}

def sample_2v2_pairs(labels, mode, n_pairs, seed):
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels, dtype=np.int64)
    by_class = indices_by_class(labels)
    if mode == "different_category_balanced":
        classes = np.asarray(sorted(by_class), dtype=np.int64)
        first_pos = rng.integers(0, len(classes), n_pairs)
        second_pos = (first_pos + rng.integers(1, len(classes), n_pairs)) % len(classes)
        first_class, second_class = classes[first_pos], classes[second_pos]
        first, second = np.empty(n_pairs, np.int64), np.empty(n_pairs, np.int64)
        for class_id in classes:
            mask = first_class == class_id
            first[mask] = rng.choice(by_class[int(class_id)], mask.sum(), replace=True)
            mask = second_class == class_id
            second[mask] = rng.choice(by_class[int(class_id)], mask.sum(), replace=True)
        return first, second
    if mode == "same_category_balanced":
        classes = np.asarray([c for c, idx in by_class.items() if len(idx) >= 2])
        chosen = rng.choice(classes, n_pairs, replace=True)
        first, second = np.empty(n_pairs, np.int64), np.empty(n_pairs, np.int64)
        for class_id in classes:
            mask = chosen == class_id
            candidates = by_class[int(class_id)]
            first[mask] = rng.choice(candidates, mask.sum(), replace=True)
            second[mask] = rng.choice(candidates, mask.sum(), replace=True)
            collisions = first[mask] == second[mask]
            selected_positions = np.flatnonzero(mask)
            while collisions.any():
                second[selected_positions[collisions]] = rng.choice(
                    candidates, collisions.sum(), replace=True
                )
                collisions = first[selected_positions] == second[selected_positions]
        return first, second
    if mode == "random_overall_trial_uniform":
        first = rng.integers(0, len(labels), n_pairs)
        second = rng.integers(0, len(labels) - 1, n_pairs)
        second += second >= first
        return first, second
    raise ValueError(f"Unknown 2v2 mode: {mode}")

def score_pairs(similarity, first, second, permutation=None):
    """Compare matched and swapped similarities, giving ties half credit."""
    columns = np.arange(similarity.shape[1]) if permutation is None else permutation
    correct = similarity[first, columns[first]] + similarity[second, columns[second]]
    swapped = similarity[first, columns[second]] + similarity[second, columns[first]]
    return float(np.mean((correct > swapped) + 0.5 * (correct == swapped)))

def evaluate_2v2(items):
    rows = []
    for item in tqdm(items, desc="2v2 jobs"):
        similarity = l2_normalize_np(item["z"]) @ l2_normalize_np(item["text_true"]).T
        for mode in TWOVTWO_MODES:
            observed_seed = derive_seed("2v2_observed", item["fold_id"], mode)
            first, second = sample_2v2_pairs(
                item["y_true"], mode, N_2V2_PAIRS, observed_seed
            )
            observed = score_pairs(similarity, first, second)
            rng = np.random.default_rng(derive_seed("2v2_null", item["fold_id"], mode))
            null = np.empty(N_2V2_PERMUTATIONS, dtype=np.float32)
            for permutation_id in range(N_2V2_PERMUTATIONS):
                permutation = rng.permutation(len(item["y_true"]))
                null_first, null_second = sample_2v2_pairs(
                    item["y_true"], mode, max(5_000, N_2V2_PAIRS // 4),
                    int(rng.integers(0, 2**31 - 1)),
                )
                null[permutation_id] = score_pairs(
                    similarity, null_first, null_second, permutation
                )
            rows.append({
                "fold_id": item["fold_id"], "condition": item["condition"],
                "run_name": item["run_name"], "checkpoint": item["checkpoint"],
                "eval_phase": item["eval_phase"], "mode": mode,
                "n_trials": len(item["y_true"]), "n_pairs": N_2V2_PAIRS,
                "n_permutations": N_2V2_PERMUTATIONS,
                "observed_2v2": observed, "null_mean": float(null.mean()),
                "null_std": float(null.std(ddof=1)),
                "z_score": float((observed - null.mean()) / (null.std(ddof=1) + 1e-12)),
                "p_value": float((1 + np.sum(null >= observed)) / (len(null) + 1)),
            })
    return pd.DataFrame(rows)
twovtwo_results = evaluate_2v2(prediction_items)
twovtwo_summary = summarize_folds(twovtwo_results, ["condition", "checkpoint", "eval_phase", "mode"], {
    "observed_mean": ("observed_2v2", "mean"), "observed_std": ("observed_2v2", fold_std),
    "null_mean": ("null_mean", "mean"), "mean_p_value": ("p_value", "mean"),
})
show_table("2v2 results by fold", twovtwo_results)
show_table("2v2 cross-validation summary", twovtwo_summary)

## 13. Compare conditions

In [ ]:
def plot_label(value, width=24):
    return "\n".join(textwrap.wrap(str(value).replace(" → ", " →\n"), width=width))

def condition_sort_key(condition):
    """Order scratch, transfer, zero-shot, and reading-test conditions."""
    text = str(condition)
    lower = text.lower()
    match = re.search(r"(\d+)%", text)
    percentage = int(match.group(1)) if match else 0
    label_mode = 0 if " true" in lower else 1 if " shuffled" in lower else 2
    if lower.startswith("imagined scratch"):
        return (0, 0, percentage, text)
    if "transfer" in lower:
        return (1, label_mode, percentage, text)
    if "zero-shot" in lower:
        return (2, label_mode, 0, text)
    if "reading test" in lower:
        return (3, label_mode, 0, text)
    return (9, label_mode, percentage, text)

def ordered_conditions(frame):
    return sorted(frame["condition"].drop_duplicates().tolist(), key=condition_sort_key)

def add_bar_values(ax, bars, means, stds, rotation=90, fontsize=7):
    labels = [
        f"{float(mean):.3f}±{float(std):.3f}"
        if np.isfinite(mean) and np.isfinite(std) else ""
        for mean, std in zip(means, stds)
    ]
    ax.bar_label(
        bars, labels=labels, padding=3, rotation=rotation,
        fontsize=fontsize, fontweight="medium",
    )

def plot_comparison(summary, mean_col, std_col, title, group_col=None,
                    group_order=None, group_labels=None, baseline_col=None, chance=None):
    """Plot fold means and standard deviations for each phase and checkpoint."""
    for (phase, checkpoint), frame in summary.groupby(["eval_phase", "checkpoint"], sort=False):
        conditions = ordered_conditions(frame)
        groups = [g for g in group_order if g in set(frame[group_col])] if group_col else [None]
        x = np.arange(len(conditions), dtype=float)
        width = 0.82 / len(groups)
        if group_col == "k":
            colors = plt.cm.viridis(np.linspace(0.12, 0.88, len(groups)))
        else:
            colors = ["#4C78A8", "#F58518", "#54A24B"][:len(groups)]
        fig, ax = plt.subplots(figsize=(max(9, 1.7 * len(conditions)), 6))
        for offset, (group, color) in enumerate(zip(groups, colors)):
            values = frame[frame[group_col] == group] if group_col else frame
            values = values.set_index("condition").reindex(conditions)
            positions = x + (offset - (len(groups) - 1) / 2) * width
            bars = ax.bar(
                positions, values[mean_col], width=width * 0.92,
                yerr=values[std_col].fillna(0), capsize=3, color=color,
                edgecolor="white", label=group_labels[group] if group_col else None,
            )
            add_bar_values(ax, bars, values[mean_col], values[std_col].fillna(0),
                           rotation=90 if group_col else 0, fontsize=7 if group_col else 8)
            if baseline_col:
                baseline_label = "Random-prior mean" if group_col == "k" else "Permutation-null mean"
                ax.scatter(positions, values[baseline_col], marker="x", s=28, color="black",
                           label=baseline_label if offset == 0 else None, zorder=4)
        if chance is not None:
            ax.axhline(chance, color="black", linestyle="--", linewidth=1,
                       alpha=0.55, label=f"Chance = {chance}")
        ax.set(title=f"{title} | {phase.title()} test | {checkpoint}", xlabel="Model condition",
               ylabel="Macro F1" if group_col is None else "Accuracy", xticks=x,
               ylim=(0, 0.20) if group_col is None else (0, 1.08))
        ax.set_xticklabels([plot_label(c) for c in conditions], rotation=18, ha="right")
        ax.grid(axis="y", alpha=0.25)
        if group_col or chance is not None:
            ax.legend(ncol=3 if group_col == "k" else 2, frameon=False)
        fig.tight_layout()
        plt.show()
        plt.close(fig)
plot_comparison(topk_summary, "topk_mean", "topk_std", "Top-k accuracy", "k", TOPK_LIST,
                {k: f"Top-{k}" for k in TOPK_LIST}, baseline_col="random_mean")
plot_comparison(macro_f1_summary, "macro_f1_mean", "macro_f1_std", "Macro F1")
plot_comparison(twovtwo_summary, "observed_mean", "observed_std", "2v2 accuracy", "mode", TWOVTWO_MODES,
                dict(zip(TWOVTWO_MODES, ["Different category", "Same category", "Random overall"])),
                baseline_col="null_mean", chance=0.5)